In [65]:
import pickle
import numpy as np
import pandas as pd
import os
import tensorflow as tf

In [66]:
twitter_data = pd.read_csv("../data/external/Twitter_Data.csv")
reddit_data = pd.read_csv("../data/external/Reddit_Data.csv")

twitter_data.rename(columns={'clean_text':"clean_comment"},inplace=True)
df = pd.concat([twitter_data,reddit_data],axis=0)

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [67]:
df.head()

,clean_comment,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [71]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [72]:
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))
to_remove_stopWords = stopwords - {'not','but','however','no','yet'}


nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [73]:
def preprocess(df:pd.DataFrame):
    #lowercase
    df['clean_comment'] = df['clean_comment'].apply(lambda x : " ".join([word.lower() for word in x.split(' ')]))
    
    # remove punctuations 
    import string
    punctuations = string.punctuation
    
    df["clean_comment"] = df["clean_comment"].str.translate(
        str.maketrans('', '', punctuations)
    )
    
    #remove stopwords
    df['clean_comment'] = df['clean_comment'].apply(lambda x:" ".join([word for word in x.split(' ') if word not in to_remove_stopWords]))
    
    #remove special characters
    import re
    df['clean_comment'] = df['clean_comment'].apply(lambda x : re.sub(r'[^a-zA-Z0-9/s?,.!]'," ", str(x)))
    
    # Lemitization 

    lemitizer  = WordNetLemmatizer()
    df['clean_comment'] = df['clean_comment'].apply(lambda x : " ".join([lemitizer.lemmatize(word) for word in x.split()]))
    
    
    return df

In [74]:
df = preprocess(df)


In [75]:
df.head()

,clean_comment,category
0,modi promised minimum government maximum gover...,-1.0
1,talk nonsense continue drama vote modi,0.0
2,say vote modi welcome bjp told rahul main camp...,1.0
3,asking supporter prefix chowkidar name modi gr...,1.0
4,answer among powerful world leader today trump...,1.0


In [76]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [77]:
x = list(df['clean_comment'].values)
y = df['category'].values


In [78]:
label_mapping = {-1:0,0:1,1:2}
y_maped = np.array([label_mapping[i] for i in y])
    

In [79]:
x_train, x_test, y_train, y_test = train_test_split(x,y_maped, test_size=0.2)

In [80]:
vectorizer = TfidfVectorizer(max_features=2000)
x_train_vectorized = vectorizer.fit_transform(x_train).toarray()
x_test_vectorized = vectorizer.transform(x_test).toarray()

In [81]:
x_train_vectorized.shape

(159766, 2000)

In [82]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu",input_shape=(x_train_vectorized.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
    
])

model.compile(optimizer="adam",loss='sparse_categorical_crossentropy', metrics=['accuracy'])


C:\Users\bobby\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [83]:
model.fit(x_train_vectorized,y_train,validation_data=epochs=5,batch_size=100)

Epoch 1/5
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.8139 - loss: 0.5087
Epoch 2/5
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8536 - loss: 0.4254
Epoch 3/5
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8644 - loss: 0.3927
Epoch 4/5
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8787 - loss: 0.3484
Epoch 5/5
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8976 - loss: 0.2933


In [84]:
loss, accuracy = model.evaluate(x_test_vectorized,y_test)

1249/1249 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8392 - loss: 0.4961


In [85]:
accuracy

0.8391668200492859

In [3]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPUs available: []


In [2]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [31]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------
# 1️⃣ Sample Data (Replace with your dataset)
# ---------------------------
texts = [
    "I love this product",
    "This is bad",
    "It is okay",
    "Amazing experience",
    "Worst purchase ever",
    "Not good not bad"
]

labels = [1, -1, 0, 1, -1, 0]   # Sentiment labels

# ---------------------------
# 2️⃣ Convert text to TF-IDF features
# ---------------------------
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts).toarray()

# Convert labels (-1,0,1) → (0,1,2)
label_mapping = {-1: 0, 0: 1, 1: 2}
y = np.array([label_mapping[l] for l in labels])

# ---------------------------
# 3️⃣ Train-Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 4️⃣ Build MLP Model
# ---------------------------
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')   # 3 classes
])

# ---------------------------
# 5️⃣ Compile Model
# ---------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# 6️⃣ Train Model
# ---------------------------
model.fit(X_train, y_train, epochs=10, batch_size=8)

# ---------------------------
# 7️⃣ Evaluate
# ---------------------------
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7500 - loss: 1.0268
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 1.0000 - loss: 0.9976
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 1.0000 - loss: 0.9702
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 1.0000 - loss: 0.9439
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 0.9175
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 1.0000 - loss: 0.8914
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 1.0000 - loss: 0.8663
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 1.0000 - loss: 0.8410
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 1.0000 - loss: 0.8154
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.7905
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.0000e+00 - loss: 1.2789
Test Accuracy: 0.0


In [46]:
df.shape

(199708, 2)

False    199708
Name: count, dtype: int64

In [52]:
reddit_data.columns[1]

'category'

In [44]:

import mlflow

mlflow.set_tracking_uri("databricks")

model_uri = 'models:/m-456c56f72b794d1aaf9d781e44a371cf'

model = mlflow.pyfunc.load_model(model_uri=model_uri)

In [45]:
import pandas as pd

In [46]:
df = pd.DataFrame(['hi how are you',"this is just a normal", 'modi id so disgusting'],columns=['clean_comment'])

In [50]:
model.predict(df)

array([0.])

In [ ]:
twitter_data.

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0
...,...,...
162975,why these 456 crores paid neerav modi not reco...,-1.0
162976,dear rss terrorist payal gawar what about modi...,-1.0
162977,did you cover her interaction forum where she ...,0.0
162978,there big project came into india modi dream p...,0.0


In [1]:
import mlflow

In [ ]:
mlflow

False

In [18]:
import mlflow

client = mlflow.tracking.MlflowClient()
print([m.name for m in client.search_registered_models()])

[]


In [6]:
import mlflow
mlflow.get_tracking_uri()

'file:///d:/CapstoneProjects/Cookie_verison/yt-comment-analyser/notebooks/mlruns'

In [5]:
import mlflow
model = mlflow.sklearn.load_model(
    "models:/youtube-model/1"
)

MlflowException: Registered Model with name=youtube-model not found

In [1]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("sqlite:///mlflow.db")
print(mlflow.get_tracking_uri())
client = MlflowClient()

# List experiments (runs)
print("Experiments:", client.search_experiments())

# List runs in default experiment
runs = client.search_runs(experiment_ids=["0"])
print("Runs found:", len(runs))

# Try to list registered models
print("Registered Models:", client.search_registered_models())

sqlite:///mlflow.db


2026/04/06 10:24:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/06 10:24:58 INFO mlflow.store.db.utils: Updating database tables
2026-04-06 10:24:58 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-04-06 10:24:58 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026-04-06 10:24:59 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
2026-04-06 10:24:59 INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2026-04-06 10:25:00 INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2026-04-06 10:25:00 INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2026-04-06 10:25:00 INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2026-04-06 10:25:01 INFO  [alembic.runtime.mig

Experiments: [<Experiment: artifact_location='file:d:/CapstoneProjects/Cookie_verison/yt-comment-analyser/notebooks/mlruns/0', creation_time=1775451318301, experiment_id='0', last_update_time=1775451318301, lifecycle_stage='active', name='Default', tags={}>]
Runs found: 0
Registered Models: []


In [4]:
mlflow.set_tracking_uri("http://52.66.145.172:5000")

In [5]:
mlflow.get_tracking_uri()

'http://52.66.145.172:5000'

In [6]:
mlflow.get_artifact_uri()

2026/04/06 10:25:55 WARNING mlflow.tracking.fluent: No active run found. A new active run will be created. If this is not intended, please create a run using `mlflow.start_run()` first.


'/home/ubuntu/artifacts/0/272429ac2b194d51aa4ddff78ef316f9/artifacts'

In [13]:
import mlflow

In [14]:
mlflow.get_tracking_uri()

'http://52.66.145.172:5000'

In [15]:
mlflow.get_artifact_uri()

's3://bobby-kumar-1950/mlflow-artifacts/0/d6d1b1b00052478fb14706b6088b5e32/artifacts'

In [23]:
mlflow.sklearn.load_model(model_uri = "s3://bobby-kumar-1950/mlflow-artifacts/1/9e58cf4a14404f49b50c9dfa883153a8/artifacts/model/model.pkl")

MlflowException: Failed to list artifacts in s3://bobby-kumar-1950/mlflow-artifacts/1/9e58cf4a14404f49b50c9dfa883153a8/artifacts/model: The AWS Access Key Id you provided does not exist in our records.

SyntaxError: invalid syntax (3821592265.py, line 1)